# 01 Data Exploration

Load the UCI diabetes readmission dataset, inspect missingness, and verify the binary target distribution.


In [ ]:
# Colab setup. Run this cell if dependencies are missing.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    # drive.mount('/content/drive')  # Optional: uncomment if storing project/data in Drive.
    !pip install -q -r requirements.txt

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('Project root:', PROJECT_ROOT)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
from src.data_download import download_dataset, colab_upload_dataset
from src.preprocessing import load_raw_csv, clean_dataframe, separate_features_target, create_splits
from src.plots import plot_class_distribution
from src.config import RAW_DATA_DIR, FIGURES_DIR

try:
    csv_path = download_dataset()
except Exception as exc:
    print(exc)
    print('If running in Colab, upload diabetic_data.csv in the next line.')
    # csv_path = colab_upload_dataset()
    raise

csv_path


In [ ]:
raw = load_raw_csv(csv_path)
print('Raw shape:', raw.shape)
raw.head()


In [ ]:
missing = raw.replace('?', float('nan')).isna().mean().sort_values(ascending=False)
missing.head(20).to_frame('missing_rate')


In [ ]:
raw['readmitted'].value_counts(dropna=False).to_frame('count')


In [ ]:
cleaned = clean_dataframe(raw)
X, y = separate_features_target(cleaned)
print('Cleaned feature shape:', X.shape)
print('Positive rate:', y.mean())
y.value_counts(normalize=True).rename('proportion').to_frame()


In [ ]:
plot_path = plot_class_distribution(y, FIGURES_DIR / 'class_distribution_all.png')
plot_path


In [ ]:
cleaned.describe(include='all').transpose().head(30)


In [ ]:
split = create_splits(X, y)
print(split.X_train.shape, split.X_val.shape, split.X_test.shape)
print('Train positive rate:', split.y_train.mean())
print('Val positive rate:', split.y_val.mean())
print('Test positive rate:', split.y_test.mean())
